# Hyperliquid trade post-mortem

Line up every closed perp trade from one wallet against the price path it lived through, then ask six questions of each trade:

1. Would it have been better **not to open** it?
2. Should it have been **stopped out**?
3. Should it have **taken profit**?
4. Should it have **kept holding**?
5. Would it have been better to **close and reopen**?
6. Would it have been better **not to add** to it?

Then replay simple rules (stops, targets, trailing stops, time stops, no adding) on the same trades, and check whether the rule that looks best on your earlier trades still helps on your later ones.

Data: fills from `userFillsByTime` and 1s/1m candles from the candle range endpoint, both on [Dwellir's Hyperliquid Index](https://www.dwellir.com/docs/hyperliquid/hyperliquid-index). Each fill and each candle returned costs 1 API credit; candles are cached in `.cache/` so re-running is free.

In [ ]:
import os, time
import pandas as pd
import matplotlib.pyplot as plt

import analysis, plots, postmortem

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
assert os.environ.get("DWELLIR_API_KEY"), "export DWELLIR_API_KEY before starting Jupyter"

WALLET = os.environ.get("WALLET", "0xYourWalletAddress")   # the wallet to review
DAYS = 30                                                   # how far back to look

## 1. Fetch fills, rebuild trades, score them

A trade starts when a position leaves zero and ends when it is back at zero. `run()` fetches the fills, rebuilds the trades, loads candles from each entry to 24 hours after each exit, and replays the rules.

In [ ]:
end = int(time.time() * 1000)
r = postmortem.run(WALLET, end - DAYS * 86_400_000, end)
df = r["df"]
postmortem.report(r)

## 2. Every trade

`mfe_pct` is the best move in your favour while the trade was open and `mae_pct` the worst move against you, both in % of your average entry. `capture` is the share of the best move you kept. `fix_*` columns are what each alternative would have added in dollars; `lesson` is the largest one.

In [ ]:
cols = ["coin", "side", "open", "hours", "notional", "adds", "net_pnl", "fees",
        "mfe_pct", "mae_pct", "realized_pct", "capture", "after_4h_close_pct",
        "fix_skip", "fix_stop", "fix_take", "fix_hold", "fix_reenter", "fix_no_adds", "lesson"]
df[cols].round(2)

## 3. Winners and losers by excursion

Trades that never moved your way sit along the bottom. If most of your losers are there, the entry is the problem and no stop or target will fix it.

In [ ]:
plots.excursion_scatter(df)
plt.show()

## 4. Where you lose money

One lesson per trade, and what fixing it would have added.

In [ ]:
display(r["lessons"].round(0))
plots.lessons_bar(r["lessons"])
plt.show()

## 5. Your worst trades, up close

Candle range in grey, entries and exits sized by fill size, the stop and target that were sized from your own trades as dashed lines.

In [ ]:
by_id = dict(enumerate(r["trades"]))
for i in df.nsmallest(3, "net_pnl").index:
    plots.trade_chart(by_id[i], r["prices"], df.loc[i], stop=df.attrs["stop"], take=df.attrs["take"])
    plt.show()

## 6. Do you exit early?

The average move after your exits, in your direction. Positive means price kept going your way after you left. A t-stat under about 2 in absolute value is indistinguishable from luck.

In [ ]:
r["exit_timing"].round(3)

## 7. Rules on the same trades

Each rule replays your real entries and exits and only steps in when it fires. Read `second_half`: a rule chosen on the earlier trades that does not help on the later ones was fitted to your history, not to your habits.

In [ ]:
display(r["rules"].round(0))
plots.rules_bar(r["rules"])
plt.show()
postmortem.pick_and_check(r["rules"])

## 8. Try your own rule

`Rule` takes `stop`, `take`, `trail` (fractions of price), `max_hours`, `no_adds` and `reenter`. Rules combine.

In [ ]:
mine = [
    analysis.Rule("stop 0.3% / take 1%", stop=0.003, take=0.01),
    analysis.Rule("no adds, max 2h", no_adds=True, max_hours=2),
]
analysis.test_rules(r["trades"], r["prices"], mine).round(0)

In [ ]:
df.to_csv("trades.csv", index=False)